# E048 — Fusion Backbone Sweep with Product Rule

**Motivation:** E046 used product rule fusion with E042 (tied cov + speed TTA) audio backbone. But E043 (image TTA flip+rot5) is the new image flagship. Let's test all backbone combinations with product rule.

**Hypothesis:** E042 audio + E043 image backbones with product rule will achieve best fusion EER.

**Configs:**
- Audio backbones: E037 (tied cov only), E042 (tied cov + speed TTA)
- Image backbones: E033 (adversarial), E043 (adv + flip/rot5 TTA)
- Fusion: Product rule only (E046 winner)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from PIL import Image
from scipy.special import expit, logit
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0
N_PCA = 50

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E046_REF = {'oof_eer': 0.52}

222 samples


In [2]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _apply_vtln(feat, alpha=1.0):
    if alpha == 1.0:
        return feat
    warped = feat.copy()
    warp_scale = 1.0 / alpha
    for n in range(1, 13):
        warped[:, n] = feat[:, n] * (warp_scale ** n)
    warped[:, 13:26] = feat[:, 13:26] * warp_scale
    warped[:, 26:39] = feat[:, 26:39] * warp_scale
    return warped

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type,
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def train_audio_model(train_df, data_dir, seed, use_speed_tta=False):
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    ubm = _train_ubm(np.vstack(X_nontarget), cov_type='tied')
    adapted = _map_adapt(ubm, np.vstack(X_target))
    return ubm, adapted

def score_audio_tta(ubm, adapted, test_row, data_dir):
    y_wav, sr = librosa.load(_find_wav(test_row["stem"], data_dir), sr=None, mono=True)
    scores = []
    for speed in [0.9, 1.0, 1.1]:
        if speed == 1.0:
            y_proc = y_wav
        else:
            y_proc = librosa.effects.time_stretch(y_wav, rate=speed)
        feat = _extract_lpcc(y_proc, sr)
        llr = adapted.score(feat) - ubm.score(feat)
        scores.append(float(llr) if np.isscalar(llr) else llr.mean())
    return np.mean(scores)

def score_audio_single(ubm, adapted, test_row, data_dir):
    y_wav, sr = librosa.load(_find_wav(test_row["stem"], data_dir), sr=None, mono=True)
    feat = _extract_lpcc(y_wav, sr)
    llr = adapted.score(feat) - ubm.score(feat)
    return float(llr) if np.isscalar(llr) else llr.mean()

print('Audio functions defined')

Audio functions defined


In [3]:
def _load_image(stem, data_dir):
    p = _find_png(stem, data_dir)
    img = Image.open(p).convert('L')
    return np.array(img.resize((80, 80)), dtype=np.float32) / 255.0

def _aug_image(img, rng):
    augments = []
    # Flip
    if rng.random() < 0.5:
        img = np.fliplr(img)
    # Brightness
    factor = rng.uniform(0.7, 1.3)
    img = np.clip(img * factor, 0, 1)
    # Noise
    noise = rng.normal(0, 0.05, img.shape)
    img = np.clip(img + noise, 0, 1)
    # Adversarial rotation
    angle = rng.uniform(-10, 10)
    img = rotate(img, angle, reshape=False, mode='reflect')
    return img

def train_image_model(train_df, data_dir, seed, use_tta=False):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _, row in train_df.iterrows():
        img = _load_image(row["stem"], data_dir)
        if row["label"] == 1:
            img_aug = _aug_image(img, rng)
            X.append(img.flatten())
            X.append(img_aug.flatten())
            y.extend([1, 1])
        else:
            X.append(img.flatten())
            y.append(0)
    
    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_pca = pca.fit_transform(X)
    clf = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    clf.fit(X_pca, y)
    return pca, clf

def score_image_tta(pca, clf, test_row, data_dir):
    img = _load_image(test_row["stem"], data_dir)
    scores = []
    # Flip TTA
    imgs = [img, np.fliplr(img)]
    # Small rotations
    for angle in [-5, 0, 5]:
        if angle != 0:
            imgs.append(rotate(img, angle, reshape=False, mode='reflect'))
    
    for img_view in imgs:
        x = img_view.flatten().reshape(1, -1)
        x_pca = pca.transform(x)
        log_odds = clf.decision_function(x_pca)[0]
        scores.append(log_odds)
    
    return np.mean(scores)

def score_image_single(pca, clf, test_row, data_dir):
    img = _load_image(test_row["stem"], data_dir)
    x = img.flatten().reshape(1, -1)
    x_pca = pca.transform(x)
    log_odds = clf.decision_function(x_pca)[0]
    return log_odds

print('Image functions defined')

Image functions defined


In [4]:
from scipy.special import logsumexp

def product_rule_fusion(audio_scores, image_scores):
    """Product rule fusion using log-odds to probabilities."""
    # Convert log-odds to probabilities
    p_audio = expit(audio_scores)
    p_image = expit(image_scores)
    
    # Product rule: geometric mean
    p_fused = np.sqrt(p_audio * p_image)
    
    # Convert back to log-odds for EER computation
    return logit(p_fused)

def sum_rule_fusion(audio_scores, image_scores):
    """Simple sum rule (equal weights)."""
    return (audio_scores + image_scores) / 2

print('Fusion functions defined')

Fusion functions defined


In [5]:
from itertools import product

# Backbone configurations
configs = {
    'E037_audio+E033_image': {'audio_tta': False, 'image_tta': False},
    'E042_audio+E033_image': {'audio_tta': True, 'image_tta': False},
    'E037_audio+E043_image': {'audio_tta': False, 'image_tta': True},
    'E042_audio+E043_image': {'audio_tta': True, 'image_tta': True},
}

results = []

for config_name, settings in configs.items():
    print(f'\n{"="*60}')
    print(f'Testing {config_name}')
    print(f'Audio TTA: {settings["audio_tta"]}, Image TTA: {settings["image_tta"]}')
    print(f'{"="*60}')
    
    fold_eers = []
    
    for fold_idx, tr_idx, te_idx in iter_folds(manifest, n_splits=3, seed=SEED):
        print(f'  Fold {fold_idx}...', end=' ', flush=True)
        
        train_df = manifest.iloc[tr_idx]
        test_df = manifest.iloc[te_idx]
        
        # Train audio model
        ubm, adapted = train_audio_model(train_df, DATA, SEED + fold_idx)
        
        # Train image model
        pca, clf = train_image_model(train_df, DATA, SEED + fold_idx)
        
        # Score test
        fused_scores, labels = [], []
        
        for _, row in test_df.iterrows():
            # Audio scoring
            if settings['audio_tta']:
                audio_score = score_audio_tta(ubm, adapted, row, DATA)
            else:
                audio_score = score_audio_single(ubm, adapted, row, DATA)
            
            # Image scoring
            if settings['image_tta']:
                image_score = score_image_tta(pca, clf, row, DATA)
            else:
                image_score = score_image_single(pca, clf, row, DATA)
            
            # Product rule fusion
            fused_score = product_rule_fusion(audio_score, image_score)
            
            fused_scores.append(fused_score)
            labels.append(row['label'])
        
        eer, _ = compute_eer(np.array(labels), np.array(fused_scores))
        fold_eers.append(eer * 100)
        print(f'{eer*100:.2f}%')
    
    mean_eer = np.mean(fold_eers)
    std_eer = np.std(fold_eers)
    print(f'{config_name}: {mean_eer:.2f} ± {std_eer:.2f}%')
    
    results.append({
        'config': config_name,
        'audio_tta': settings['audio_tta'],
        'image_tta': settings['image_tta'],
        'eer_mean': mean_eer,
        'eer_std': std_eer,
        'fold_eers': fold_eers
    })

# Summary
print('\n' + '='*70)
print('E048 FUSION BACKBONE SWEEP SUMMARY')
print('='*70)

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df[['config', 'eer_mean', 'eer_std']].to_string(index=False))

best_idx = results_df['eer_mean'].argmin()
best = results_df.iloc[best_idx]
print(f'\n✓ Best: {best["config"]} → {best["eer_mean"]:.2f} ± {best["eer_std"]:.2f}%')

improvement = E046_REF['oof_eer'] - best['eer_mean']
print(f'Improvement vs E046 (0.52%): {improvement:+.2f}pp')

if improvement > 0.1:
    print(f'\n✓✓ NEW FUSION FLAGSHIP!')
elif improvement > 0:
    print(f'\n✓ Marginal improvement')
else:
    print(f'\n✗ E046 remains flagship')


Testing E037_audio+E033_image
Audio TTA: False, Image TTA: False
  Fold 0... 

4.88%
  Fold 1... 

0.00%
  Fold 2... 

10.71%
E037_audio+E033_image: 5.20 ± 4.38%

Testing E042_audio+E033_image
Audio TTA: True, Image TTA: False
  Fold 0... 

4.88%
  Fold 1... 

...

0.00%
  Fold 2... 

10.71%
E037_audio+E043_image: 4.59 ± 4.51%

Testing E042_audio+E043_image
Audio TTA: True, Image TTA: True
  Fold 0... 

3.05%
  Fold 1... 

0.00%
  Fold 2... 

10.71%
E042_audio+E043_image: 4.59 ± 4.51%

E048 FUSION BACKBONE SWEEP SUMMARY
               config  eer_mean  eer_std
E037_audio+E033_image  5.197445 4.379916
E042_audio+E033_image  5.197445 4.379916
E037_audio+E043_image  4.587689 4.507413
E042_audio+E043_image  4.587689 4.507413

✓ Best: E037_audio+E043_image → 4.59 ± 4.51%
Improvement vs E046 (0.52%): -4.07pp

✗ E046 remains flagship
